# 02 · Baseline v1: CLIP bias on FairFace (gender, race, age)

Pretrained CLIP ViT-B/32, **no mitigation**. For every query we rank the whole FairFace
validation split (~10.9k faces) and measure how the top-k is distributed over gender,
race and age.

**What is where:** the logic lives in the repo (`src/vlbias/`), this notebook only runs it
and looks at the results. Every setting comes from `configs/experiments/baseline_fairface.yaml`.

**Colab:** Runtime → Change runtime type → **T4 GPU**. A full run takes a few minutes, mostly
downloading FairFace and encoding the images once.

## 0. Setup

Clones your repo and installs it. If the repo is **private**, add a GitHub token as a Colab
secret first: key icon in the left sidebar → *Add new secret* → name `GITHUB_TOKEN`, value
= your token → toggle *Notebook access* on.

In [ ]:
import os, sys

REPO = "simalyucel/clip-bias-mitigation"
ROOT = "/content/clip-bias-mitigation"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")      # only needed for a private repo
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if not os.path.exists(ROOT):
    !git clone -q {url} {ROOT}
else:
    !git -C {ROOT} pull -q
%cd {ROOT}
!pip -q install open_clip_torch datasets
!pip -q install -e .
sys.path.insert(0, f"{ROOT}/src")

In [ ]:
# the unit tests should all pass before trusting any number
!python -m pytest -q

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path

from vlbias.utils.config import load_yaml, set_seed
from vlbias.models.load import load_model
from vlbias.data.queries import load_queries
from vlbias.data.fairface import load_fairface
from vlbias.retrieval.embed import gallery_embeddings, encode_texts
from vlbias.retrieval.rank import top_k_indices
from vlbias.baseline import evaluate, summarize

pd.set_option("display.precision", 3)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch to a T4!")

**Optional:** cache the image embeddings on Google Drive, so a new Colab session does not
re-encode FairFace. Set `USE_DRIVE = True`.

In [ ]:
USE_DRIVE = False

cfg = load_yaml("configs/experiments/baseline_fairface.yaml")
CACHE_DIR = Path(cfg["cache_dir"])
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/thesis/embeddings")
set_seed(cfg["seed"])
cfg

## 1. Model and queries

In [ ]:
vlm = load_model(load_yaml(cfg["model"]))
queries = [q for path in cfg["query_sets"] for q in load_queries(path)]

qdf = pd.DataFrame([q.to_dict() for q in queries])
print(f"{len(queries)} queries")
display(qdf.groupby(["query_set", "category"]).agg(concepts=("concept", "nunique"),
                                                    templates=("template", "nunique"),
                                                    queries=("text", "count")))
qdf.sample(8, random_state=0)[["query_set", "text"]]

## 2. The gallery: FairFace validation split

Look at the composition first. It decides how to read the **uniform** reference: if a group
is rare in the gallery, a top-k can never be uniform for it, whatever the model does.

In [ ]:
ds = cfg["dataset"]
gallery = load_fairface(ds["split"], str(ds["padding"]), ds["max_images"], cfg["seed"])
print(gallery.name, len(gallery), "images")

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5), gridspec_kw={"width_ratios": [2, 7, 9]})
for ax, attr in zip(axes, cfg["attributes"]):
    comp = gallery.composition(attr)
    ax.bar(list(comp), list(comp.values()), color="#4C72B0")
    ax.axhline(1 / len(comp), ls="--", c="k", lw=1, label="uniform")
    ax.set_title(attr); ax.tick_params(axis="x", rotation=45); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Embeddings

Images are encoded once (and cached), the queries in a second. After this, everything is
just matrix products on the embeddings, which is also how mitigation methods will plug in.

In [ ]:
g_emb = gallery_embeddings(vlm, gallery, CACHE_DIR, batch_size=cfg["batch_size"])
q_emb = encode_texts(vlm, [q.text for q in queries])
g_emb.shape, q_emb.shape

## 4. Run the evaluation

For every query × attribute × reference × k: MaxSkew, MinSkew, max|Skew|, NDKL, plus the
Skew of every single group. Saved as CSV in `results/`.

In [ ]:
labels = {a: gallery.labels[a] for a in cfg["attributes"]}
m = cfg["metrics"]
per_query, per_group = evaluate(q_emb, queries, g_emb, labels, gallery.label_names,
                                ks=m["k"], references=m["references"],
                                zero_floor=m["zero_floor"], custom_references=m.get("custom"))
for df in (per_query, per_group):
    df.insert(0, "model", vlm.name); df.insert(1, "dataset", gallery.name)
summary = summarize(per_query)

out = Path(cfg["output_dir"]); out.mkdir(parents=True, exist_ok=True)
per_query.to_csv(out / "per_query.csv", index=False)
per_group.to_csv(out / "per_group.csv", index=False)
summary.to_csv(out / "summary.csv", index=False)
print(len(per_query), "per-query rows,", len(per_group), "per-group rows")
summary.head()

## 5. Sanity check against Chuang et al. (2023)

Same setup as their Table 5: 10 concepts, one template, MaxSkew@1000, uniform reference,
FairFace validation. Their CLIP ViT-B/32 zero-shot numbers: **gender .206, race .743, age .797**.

Close values (roughly within ±0.05) mean the pipeline is right. Small differences can come
from the FairFace crop (`padding`), image preprocessing or library versions.

In [ ]:
paper = {"gender": 0.206, "race": 0.743, "age": 0.797}
s = summary.query("query_set == 'chuang2023' and reference == 'uniform' and k == 1000")
check = s.set_index("attribute")[["max_skew_mean"]].rename(columns={"max_skew_mean": "ours"})
check["Chuang et al."] = pd.Series(paper)
check["difference"] = check["ours"] - check["Chuang et al."]
check

## 6. Uniform vs gallery reference

* **uniform** = "every group should be equally represented". It mixes the model's bias with
  the gallery's own imbalance.
* **gallery** = "the top-k should look like the collection". It isolates what the **model**
  adds (Kong et al. 2023 call the other part test-time bias).

Expect the biggest gap for **age**: FairFace is mostly 20-39 year olds, so even a perfectly
neutral model gets a high MaxSkew against a uniform reference.

In [ ]:
tab = (summary.query("k == 1000")
       .pivot_table(index=["query_set", "attribute"], columns="reference",
                    values=["max_skew_mean", "ndkl_mean"]))
tab

## 7. MaxSkew hides the worst case

MaxSkew is bounded above (a group can at most fill the whole top-k), MinSkew is not (a group
can disappear). So MinSkew and NDKL often tell a stronger story than MaxSkew alone.

In [ ]:
(summary.query("reference == 'gallery'")
 .pivot_table(index=["query_set", "attribute"], columns="k",
              values=["max_skew_mean", "min_skew_mean", "ndkl_mean"]))

## 8. Who is favoured, for which kind of concept?

Mean Skew per group (gallery reference, k = 100), split by concept category.
Red = over-represented, blue = under-represented. The interesting question: are some groups
systematically pulled up by *negative* concepts and others by *positive* ones?

In [ ]:
def skew_heatmap(attr, k=100, reference="gallery"):
    d = per_group.query("attribute == @attr and k == @k and reference == @reference")
    piv = d.pivot_table(index="category", columns="group", values="skew", aggfunc="mean")
    piv = piv[gallery.label_names[attr]]
    lim = np.nanmax(np.abs(piv.values))
    fig, ax = plt.subplots(figsize=(1.1 * piv.shape[1] + 2, 0.6 * piv.shape[0] + 1.2))
    im = ax.imshow(piv.values, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=45, ha="right")
    ax.set_yticks(range(piv.shape[0]), piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.values[i, j]:+.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label="mean Skew"); ax.set_title(f"{attr}: mean Skew@{k} ({reference} ref.)")
    plt.tight_layout(); plt.show()

for attr in cfg["attributes"]:
    skew_heatmap(attr)

## 9. Occupations and gender

Skew of *Female* per occupation (mean over the 3 templates, gallery reference, k = 100).
Positive = more women than in the gallery. Remember: nobody in FairFace has an occupation,
so this is purely the association between the occupation word and facial appearance.

In [ ]:
def occupation_plot(attr="gender", group="Female", k=100):
    d = per_group.query("query_set == 'occupations' and attribute == @attr and group == @group "
                        "and k == @k and reference == 'gallery'")
    s = d.groupby("concept")["skew"].mean().sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(s) + 1))
    ax.barh(s.index, s.values, color=np.where(s.values > 0, "#C44E52", "#4C72B0"))
    ax.axvline(0, c="k", lw=0.8)
    ax.set_xlabel(f"Skew@{k} of '{group}' (gallery reference)"); ax.set_title(f"Occupations: {attr}")
    plt.tight_layout(); plt.show()

occupation_plot("gender", "Female")

In [ ]:
# same view for race: pick any group, e.g. "Black" or "White"
occupation_plot("race", "Black")

## 10. How much does the wording matter?

For each concept in Berg et al.'s set, the spread of MaxSkew across its 8 templates
(gender, gallery reference, k = 100). Long bars = numbers that depend heavily on phrasing,
which is why we average over templates.

In [ ]:
d = per_query.query("query_set == 'berg2022_test' and attribute == 'gender' "
                    "and reference == 'gallery' and k == 100")
spread = d.groupby("concept")["max_skew"].agg(["mean", "std", "min", "max"]).sort_values("mean")
fig, ax = plt.subplots(figsize=(7, 0.25 * len(spread) + 1))
ax.hlines(spread.index, spread["min"], spread["max"], color="#999", lw=3)
ax.plot(spread["mean"], spread.index, "o", color="k", ms=4)
ax.set_xlabel("MaxSkew@100 (gender): min - mean - max over templates")
plt.tight_layout(); plt.show()
print("mean std across templates:", round(spread["std"].mean(), 3))

## 11. Look at the images

Top results for any query, with their labels.

In [ ]:
def show(query, k=12):
    from vlbias.data.queries import fill_template
    e = encode_texts(vlm, [query])
    top = top_k_indices(e, g_emb, k)[0]
    fig, axes = plt.subplots(2, k // 2, figsize=(2 * k // 2, 5))
    for ax, i in zip(axes.flat, top):
        ax.imshow(gallery.images[int(i)]["image"]); ax.axis("off")
        ax.set_title("\n".join(gallery.label_names[a][gallery.labels[a][i]] for a in cfg["attributes"]),
                     fontsize=7)
    fig.suptitle(f'"{query}"'); plt.tight_layout(); plt.show()

show("A photo of a criminal person")
show("a photo of a person who works as a nurse")

## 12. Save the results

Colab files disappear when the session ends. Download the CSVs (or copy them to Drive).
Raw results stay out of git; later we can commit a small summary table.

In [ ]:
!cd {cfg["output_dir"]} && zip -q -r /content/baseline_fairface_results.zip .
from google.colab import files
files.download("/content/baseline_fairface_results.zip")

## What to note for the report

* Section 5: does the replication match Chuang et al.? (If yes, the pipeline is validated.)
* Section 6: how much of the "bias" against a uniform reference is just the gallery (age!)?
* Sections 7-9: the strongest skews, and which groups are favoured by negative vs positive concepts.
* Section 10: how sensitive the numbers are to wording.

**Next:** retrieval utility (Recall@K on Flickr30K / MS-COCO), so that every mitigation
method can be judged on fairness *and* on how much it hurts retrieval.